<a href="https://colab.research.google.com/github/wanadzhar913/bank-transaction-classification-debertav3/blob/master/notebooks/02_model_building_modernbert_text_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip3 install datasets huggingface_hub flash-attn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 121.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [6]:
# import transformers
# import datasets
# import huggingface_hub
# import sklearn
# import torch
# import numpy as np
# import pandas as pd
# import flash_attn

# print(f'transformers version: {transformers.__version__}')
# print(f'datasets version: {datasets.__version__}')
# print(f'huggingface_hub version: {huggingface_hub.__version__}')
# print(f'scikit-learn version: {sklearn.__version__}')
# print(f'torch version: {torch.__version__}')
# print(f'numpy version: {np.__version__}')
# print(f'pandas version: {pd.__version__}')
# print(f'flast attention version: {flash_attn.__version__}')

transformers version: 4.55.2
datasets version: 4.0.0
huggingface_hub version: 0.34.4
scikit-learn version: 1.6.1
torch version: 2.8.0+cu126
numpy version: 2.0.2
pandas version: 2.2.2
flast attention version: 2.8.3


In [40]:
import os
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from datasets import load_dataset, Dataset
from huggingface_hub import create_repo, notebook_login
from sklearn.metrics import classification_report, average_precision_score, \
                            accuracy_score, f1_score, precision_score, recall_score

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoConfig, pipeline, \
                         ModernBertForSequenceClassification, \
                         DataCollatorWithPadding

pd.set_option('display.max_colwidth', None)
%matplotlib inline

In [8]:
notebook_login()

### 1.0 Load dataset & tokenizer

In [9]:
!mkdir data/
!cd data/ && wget https://raw.githubusercontent.com/wanadzhar913/bank-transaction-classification/refs/heads/master/data/train.csv -q
!cd data/ && wget https://raw.githubusercontent.com/wanadzhar913/bank-transaction-classification/refs/heads/master/data/test.csv -q

In [10]:
ds = load_dataset("csv", data_files={"train": "data/train.csv", "test": "data/test.csv"})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [19]:
ds

DatasetDict({
    train: Dataset({
        features: ['client_id', 'bank_id', 'account_id', 'txn_id', 'txn_date', 'description', 'amount', 'description_stem', 'description_clean_len', 'day_monday', 'day_friday', 'day_weekend', 'monthly_transaction_count', 'monthly_transaction_count_loans', 'monthly_transaction_count_transfer_credit', 'monthly_transaction_count_transfer_deposit', 'monthly_transaction_count_payroll', 'monthly_transaction_count_uncategorized', 'monthly_transaction_count_restaurants', 'monthly_transaction_count_check_deposit', 'monthly_transaction_count_third_party', 'monthly_transaction_count_food_and_beverage_services', 'monthly_transaction_count_internal_account_transfer', 'monthly_transaction_count_shops', 'monthly_transaction_count_supermarkets_and_groceries', 'monthly_transaction_count_telecommunication_services', 'monthly_transaction_count_bank_fees', 'monthly_transaction_count_utilities', 'monthly_transaction_count_insurance', 'monthly_transaction_count_digital_ent

In [11]:
# ensure categories are identical across splits
train_cats = sorted(set(ds["train"]["category"]))
test_cats  = sorted(set(ds["test"]["category"]))

assert set(train_cats) == set(test_cats), "Num. of categories are different between train/test"

In [12]:
test_cats

['ATM',
 'Arts and Entertainment',
 'Bank Fee',
 'Bank Fees',
 'Check Deposit',
 'Clothing and Accessories',
 'Convenience Stores',
 'Department Stores',
 'Digital Entertainment',
 'Food and Beverage Services',
 'Gas Stations',
 'Gyms and Fitness Centers',
 'Healthcare',
 'Insurance',
 'Interest',
 'Internal Account Transfer',
 'Loans',
 'Payment',
 'Payroll',
 'Restaurants',
 'Service',
 'Shops',
 'Supermarkets and Groceries',
 'Tax Refund',
 'Telecommunication Services',
 'Third Party',
 'Transfer',
 'Transfer Credit',
 'Transfer Debit',
 'Transfer Deposit',
 'Travel',
 'Uncategorized',
 'Utilities']

In [14]:
# label mapping
sorted_categories = sorted(train_cats)

label2id = {c: i for i, c in enumerate(sorted_categories)}
id2label = {i: c for c, i in label2id.items()}

In [16]:
def map_category(example):
    example["labels"] = label2id.get(example["category"], -1)
    return example

In [17]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [18]:
def tokenize_fn(batch):
    return tokenizer(batch["description"], truncation=True)

In [20]:
ds = ds.map(map_category)

Map:   0%|          | 0/199260 [00:00<?, ? examples/s]

Map:   0%|          | 0/49816 [00:00<?, ? examples/s]

In [22]:
ds = ds.map(tokenize_fn, batched=True, remove_columns=[c for c in ds["train"].column_names if c not in ["input_ids","attention_mask","labels"]])

Map:   0%|          | 0/199260 [00:00<?, ? examples/s]

Map:   0%|          | 0/49816 [00:00<?, ? examples/s]

In [23]:
ds

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 199260
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 49816
    })
})

In [36]:
ds["train"][0]

{'labels': 15,
 'input_ids': [50281,
  30430,
  6022,
  272,
  3700,
  432,
  5642,
  44,
  721,
  28757,
  11204,
  2683,
  318,
  4,
  1638,
  1099,
  50282],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [27]:
# DataLoaders with dynamic padding
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [31]:
train_loader = DataLoader(ds["train"], batch_size=64, shuffle=True, collate_fn=collator)
val_loader   = DataLoader(ds["test"],  batch_size=64, shuffle=False, collate_fn=collator)

In [34]:
# Inspect `train_loader`
# for i, batch in enumerate(train_loader):
#     print(batch["input_ids"].shape, batch["attention_mask"].shape)
#     if i == 0:  # only look at the first batch
#         print(tokenizer.batch_decode(batch["input_ids"][:5], skip_special_tokens=True))
#         break

torch.Size([64, 53]) torch.Size([64, 53])
['DEBIT CARD PURCHASE AT Maryse Hemant SER, Maryse Hemant, FL ON 080523 FROM CARD#: 3168', 'KEEP THE CHANGE TRANSFER TO ACCT 1748 FOR 09/01/23', 'Withdrawal Visa Debit / SHEETZ ECOMM6115 1036 PA Date 07/06/23 04164070000055603556030 Card 7052', 'SPEEDWAY 2505 US-1', 'APPLE CASH SENT MONEY 1INFINITELOOPCA']


### 2.0 Loading the `ModernBERT` model

In [38]:
config = AutoConfig.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=len(sorted_categories),
    id2label=id2label,
    label2id=label2id
)

config.json: 0.00B [00:00, ?B/s]

In [42]:
model = ModernBertForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    config=config,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertUnpaddedRotaryEmbedding(dim=64, base=160000.0, scale_base=None)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, 

### 3.0 Load Optimizer & Scheduler

In [44]:
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=1e-5, eps=1e-8, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5, verbose=True)

TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

### Resources

- https://medium.com/data-and-beyond/complete-guide-to-building-bert-model-from-sratch-3e6562228891
- https://huggingface.co/answerdotai/ModernBERT-base
- https://huggingface.co/blog/davidberenstein1957/fine-tune-modernbert-on-synthetic-data